# 02 — Feature Analysis for Encoding Success

**Purpose:** Analyze which EEG features best discriminate successfully encoded
(later recalled) vs. forgotten items. This guides feature selection for the classifier.

**Key hypothesis:** Alpha/beta desynchronization and frontal theta increase
predict encoding success (Duan et al. 2025).

In [ ]:
import sys
sys.path.insert(0, "../..")

import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

from classifiers.encoding.data import load_subject_data, PEERSConfig
from classifiers.encoding.features import (
    EncodingFeatureConfig,
    compute_alpha_beta_desync,
    compute_theta_power,
)
from shared.features.spectral import compute_band_power, SpectralConfig

PEERS_DIR = Path("../../data/raw/peers")
mne.set_log_level("WARNING")
plt.style.use("dark_background")

## 1. Load Data and Split by Recall

In [ ]:
# Load subject data
from shared.preprocessing.bids import get_bids_subjects
subjects = get_bids_subjects(PEERS_DIR)
config = PEERSConfig(bids_root=PEERS_DIR)

dataset = load_subject_data(PEERS_DIR, subject=subjects[0], config=config)

if dataset is not None:
    epochs = dataset.epochs
    labels = dataset.labels

    recalled_idx = np.where(labels == 1)[0]
    forgotten_idx = np.where(labels == 0)[0]

    print(f"Total epochs: {len(epochs)}")
    print(f"Recalled: {len(recalled_idx)}, Forgotten: {len(forgotten_idx)}")
else:
    print("Load PEERS data first: python data/download_peers.py")

## 2. Alpha/Beta Desynchronization — Recalled vs. Forgotten

The key encoding signature: alpha/beta power DECREASES more for items
that are later recalled. More desynchronization = better encoding.

In [ ]:
# Compute alpha/beta desynchronization
feat_config = EncodingFeatureConfig()
ab_desync = compute_alpha_beta_desync(epochs, feat_config)

# Compare recalled vs forgotten
desync_recalled = ab_desync[recalled_idx].mean(axis=1)  # mean across channels
desync_forgotten = ab_desync[forgotten_idx].mean(axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Box plot
data_to_plot = [desync_recalled, desync_forgotten]
bp = ax1.boxplot(data_to_plot, labels=["Recalled", "Forgotten"],
                 patch_artist=True, widths=0.5)
bp["boxes"][0].set_facecolor("#4FC3F7")
bp["boxes"][1].set_facecolor("#E57373")
for box in bp["boxes"]:
    box.set_edgecolor("#333")
    box.set_linewidth(2)
ax1.set_ylabel("α/β Desynchronization (%)")
ax1.set_title("α/β Desync: Recalled vs Forgotten")

# Statistical test
t_stat, p_val = stats.ttest_ind(desync_recalled, desync_forgotten)
ax1.text(0.5, 0.95, f"t={t_stat:.2f}, p={p_val:.4f}",
         transform=ax1.transAxes, ha="center", fontsize=10,
         color="#FFD54F" if p_val < 0.05 else "#999")

# Histogram
ax2.hist(desync_recalled, bins=20, alpha=0.6, label="Recalled", color="#4FC3F7",
         edgecolor="#333", linewidth=1)
ax2.hist(desync_forgotten, bins=20, alpha=0.6, label="Forgotten", color="#E57373",
         edgecolor="#333", linewidth=1)
ax2.set_xlabel("α/β Desynchronization (%)")
ax2.set_ylabel("Count")
ax2.set_title("Distribution")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Recalled mean desync: {desync_recalled.mean():.2f}%")
print(f"Forgotten mean desync: {desync_forgotten.mean():.2f}%")
print(f"Difference: {desync_recalled.mean() - desync_forgotten.mean():.2f}%")

## 3. Theta Power — Frontal Memory Engagement

Theta power should increase in frontal channels during successful encoding,
reflecting hippocampal-frontal memory circuit activation.

In [ ]:
# Compute theta power
theta = compute_theta_power(epochs, feat_config)

# Frontal channels only
from classifiers.encoding.features import FRONTAL_CHANNELS
frontal_idx = [epochs.ch_names.index(ch) for ch in FRONTAL_CHANNELS
               if ch in epochs.ch_names]

if frontal_idx:
    theta_frontal = theta[:, frontal_idx].mean(axis=1)
    theta_recalled = theta_frontal[recalled_idx]
    theta_forgotten = theta_frontal[forgotten_idx]

    fig, ax = plt.subplots(figsize=(8, 5))
    bp = ax.boxplot([theta_recalled, theta_forgotten],
                    labels=["Recalled", "Forgotten"],
                    patch_artist=True, widths=0.5)
    bp["boxes"][0].set_facecolor("#4FC3F7")
    bp["boxes"][1].set_facecolor("#E57373")
    for box in bp["boxes"]:
        box.set_edgecolor("#333")
        box.set_linewidth(2)

    t_stat, p_val = stats.ttest_ind(theta_recalled, theta_forgotten)
    ax.set_ylabel("Frontal Theta Power (μV²/Hz)")
    ax.set_title(f"Frontal Theta: Recalled vs Forgotten (t={t_stat:.2f}, p={p_val:.4f})")
    plt.tight_layout()
    plt.show()

    print(f"Frontal channels used: {[epochs.ch_names[i] for i in frontal_idx]}")
    print(f"Recalled mean: {theta_recalled.mean():.4f}")
    print(f"Forgotten mean: {theta_forgotten.mean():.4f}")
else:
    print("No frontal channels found in this montage")

## 4. Band Power Comparison Across All Bands

Compare spectral power in all frequency bands between recalled and forgotten items.

In [ ]:
# Band power for all bands, averaged across channels
active_epochs = epochs.copy().crop(tmin=0.0, tmax=0.6)
powers = compute_band_power(active_epochs, SpectralConfig(normalize=False))

band_names = sorted(powers.keys())
recalled_means = []
forgotten_means = []

for band in band_names:
    band_mean = powers[band].mean(axis=1)  # avg across channels
    recalled_means.append(band_mean[recalled_idx].mean())
    forgotten_means.append(band_mean[forgotten_idx].mean())

x = np.arange(len(band_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, recalled_means, width, label="Recalled",
               color="#4FC3F7", edgecolor="#333", linewidth=2)
bars2 = ax.bar(x + width/2, forgotten_means, width, label="Forgotten",
               color="#E57373", edgecolor="#333", linewidth=2)

ax.set_xlabel("Frequency Band")
ax.set_ylabel("Power (μV²/Hz)")
ax.set_title("Band Power: Recalled vs Forgotten (0-600ms)")
ax.set_xticks(x)
ax.set_xticklabels([b.capitalize() for b in band_names])
ax.legend()

plt.tight_layout()
plt.show()

# Print with effect sizes
print(f"\n{'Band':<10} {'Recalled':>12} {'Forgotten':>12} {'Diff':>10}")
print("-" * 48)
for band, r, f in zip(band_names, recalled_means, forgotten_means):
    print(f"{band:<10} {r:>12.6f} {f:>12.6f} {r-f:>10.6f}")